In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 29. text text text text

> **Learning Goals**
> - text text(TP)text Matrixtext text text text
> - text text(PP)text text(bubble) Problemtext text
> - 1F1B text text

## 29.1 text Model text text

Data text: Modeltext text GPUtext text text. 70B+ Modeltext text text.

text:
- **text text (TP)**: text text text text GPUtext text
- **text text (PP)**: text text GPUtext text
- **3D text**: DP + TP + PP text (GPT-3 Training text)

## 29.2 text text (Megatron-LM text)

text $Y = XW$text text:

### Column Parallel (text text)
$W = [W_1; W_2]$ (text text)
$$Y = XW = X[W_1; W_2] = [XW_1, XW_2]$$
- text GPUtext $W_i$ Calculation, Result $XW_i$ text
- Outputtext text All-Gather

### Row Parallel (text text)
$W = [W_1, W_2]$ (text text)
$$Y = XW = [X_1, X_2][W_1; W_2] = X_1 W_1 + X_2 W_2$$
- text GPUtext $X_i W_i$ Calculation
- All-Reducetext text

Megatron: Attentiontext column, FFNtext column + row text All-Reduce 1text.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Column Parallel Linear text
class ColumnParallelLinear(nn.Module):
    """Wtext text text (text text)."""
    def __init__(self, in_features, out_features, n_gpus=2):
        super().__init__()
        assert out_features % n_gpus == 0
        self.n_gpus = n_gpus
        self.out_per_gpu = out_features // n_gpus
        # text GPUtext Weight
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(in_features, self.out_per_gpu) * 0.1)
            for _ in range(n_gpus)
        ])

    def forward(self, x):
        # text GPUtext text Subspace Computation
        outputs = [x @ w for w in self.weights]
        # All-Gather: Resulttext text
        return torch.cat(outputs, dim=-1)

# Row Parallel Linear
class RowParallelLinear(nn.Module):
    """Wtext text text (Inputdegrees text)."""
    def __init__(self, in_features, out_features, n_gpus=2):
        super().__init__()
        assert in_features % n_gpus == 0
        self.n_gpus = n_gpus
        self.in_per_gpu = in_features // n_gpus
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(self.in_per_gpu, out_features) * 0.1)
            for _ in range(n_gpus)
        ])

    def forward(self, x):
        # Inputdegrees text
        x_chunks = torch.chunk(x, self.n_gpus, dim=-1)
        # text GPU Computation
        partial = [xc @ w for xc, w in zip(x_chunks, self.weights)]
        # All-Reduce: Sumtext
        return sum(partial)

# Test
d_in, d_out = 64, 64
n_gpus = 2

# Original
torch.manual_seed(0)
W_full = torch.randn(d_in, d_out) * 0.1

# Column parallel text
col = ColumnParallelLinear(d_in, d_out, n_gpus)
with torch.no_grad():
    # Weighttext W_fulltext text Setting
    for i in range(n_gpus):
        col.weights[i].copy_(W_full[:, i*col.out_per_gpu:(i+1)*col.out_per_gpu])

x = torch.randn(4, d_in)
y_full = x @ W_full
y_col = col(x)
print(f"Column Parallel Error: {(y_full - y_col).abs().max():.2e}")

# Row parallel
row = RowParallelLinear(d_in, d_out, n_gpus)
with torch.no_grad():
    for i in range(n_gpus):
        row.weights[i].copy_(W_full[i*row.in_per_gpu:(i+1)*row.in_per_gpu, :])
y_row = row(x)
print(f"Row Parallel Error: {(y_full - y_row).abs().max():.2e}")
print("\n=> text text text SumtextPlane Originaltext text!")


## 29.3 Attentiontext text text

MHAtext text TPtext text: text text text GPUtext text.

- $h$text text, $n$ GPU → text GPUtext $h/n$text text
- text $W_i^Q, W_i^K, W_i^V$ Calculation (text)
- text Result Concat → All-Reducetext text

## 29.4 text text

text text GPUtext text:
- GPU 0: text 1-4
- GPU 1: text 5-8
- GPU 2: text 9-12

Forward Passtext GPU 0 → 1 → 2 text. Backpropagationtext text.

### text Problem
text GPUtext text text text GPUtext text → text Time(text) text.

text text: $\frac{p-1}{m + p - 1}$
- $p$: text text text
- $m$: text text

text: **text** — text $m$text text text text.


In [ ]:
# text text Visualization
def pipeline_bubble_ratio(p, m):
    """Bubble Ratio."""
    return (p - 1) / (m + p - 1)

# text p, mtext text
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bubble Ratio
ax = axes[0]
for p in [2, 4, 8, 16]:
    ms = np.arange(1, 50)
    ratios = [pipeline_bubble_ratio(p, m) for m in ms]
    ax.plot(ms, ratios, label=f'p={p}', linewidth=2)
ax.set_xlabel('textBatch text m')
ax.set_ylabel('Bubble Ratio')
ax.set_title('Pipeline Bubble Ratio')
ax.legend(); ax.grid(True, alpha=0.3)

# Gantt text (text)
ax = axes[1]
p, m = 4, 8
# text GPUtext text text (Forward Pass F, Backward Pass B, text .)
schedule = []
for stage in range(p):
    s = []
    # forward: stage text
    for micro in range(m):
        # text Time = stage + micro
        for t in range(stage + micro + 1):
            pass
    # text forward text backward
    s = ['.'] * (2 * m + 2 * (p - 1))
    # forward Pattern
    for micro in range(m):
        start = stage + micro
        if start < len(s):
            s[start] = f'F{micro}'
    # backward
    for micro in range(m):
        start = m + 2 * (p - 1) - stage + micro
        if 0 <= start < len(s):
            s[start] = f'B{micro}'
    schedule.append(s)

# Visualization (text)
for i, s in enumerate(schedule):
    for j, x in enumerate(s):
        if x.startswith('F'):
            ax.barh(i, 1, left=j, color='blue', alpha=0.7)
        elif x.startswith('B'):
            ax.barh(i, 1, left=j, color='red', alpha=0.7)
        else:
            ax.barh(i, 1, left=j, color='gray', alpha=0.3)
ax.set_xlabel('Time Step')
ax.set_ylabel('GPU (stage)')
ax.set_title(f'Pipeline text (p={p}, m={m})')
ax.set_yticks(range(p))
plt.tight_layout()
plt.savefig('../figures/ch29_pipeline_bubble.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"p=4, m=8 text text: {pipeline_bubble_ratio(4, 8)*100:.1f}%")
print(f"p=4, m=32 Bubble Ratio: {pipeline_bubble_ratio(4, 32)*100:.1f}%")
print("=> textBatch text text Bubble Decrease.")


## 29.5 1F1B text

text text: text forward text text backward → text text.

**1F1B**: forward 1text, backward 1text text. textValue text text.


## 29.6 3D text (DP + TP + PP)

GPT-3 (175B) Training text:
- DP: Data text (text text)
- TP: Attention/FFN text text
- PP: text text

text: 1024 GPU = 8 DP × 8 TP × 16 PP

## 29.7 Key Takeaways

| text | text text | text | text |
|---|---|---|---|
| Data (DP) | Data | All-Reduce (text) | Model text |
| text (TP) | text Matrix | All-Reduce / All-Gather | text text |
| text (PP) | text | Point-to-Point | text text + textValue |
| 3D | text | text | text |

## Exercises

1. Column paralleltext row paralleltext text text text Resulttext text text.
2. text text text p=2, 4, 8text m=4, 16, 64text text Calculationtext.
3. Attention text 4 GPUtext text text text text.
4. 3D text 1024 GPUtext DP=8, TP=8, PP=16text Contentstext text text.
5. 1F1B text text text text text.

> Solutions: `solutions/ch29_solutions.ipynb`
